# TF-IDF Retrieval Results

This notebook analyzes the first retrieval results obtained with the TF-IDF baseline.

The goal is not only to display rankings, but also to understand:
- when TF-IDF works well;
- when it retrieves only partial matches;
- when lexical overlap is misleading;
- how the results can be compared later with transformer-based embeddings.

In [1]:
import sys
from pathlib import Path

import pandas as pd

In [2]:
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

PROJECT_ROOT

WindowsPath('c:/dev/nlp-theses-search')

In [3]:
from src.tfidf_search import load_data, build_tfidf_matrix, search_tfidf

In [4]:
DATA_PATH = PROJECT_ROOT / "data" / "processed" / "theses_clean.csv"

df = load_data(DATA_PATH)

df.shape, df.columns.tolist()

((18242, 11),
 ['id',
  'title',
  'title_en',
  'abstract',
  'year',
  'discipline',
  'subjects',
  'institution',
  'status',
  'url',
  'text'])

In [5]:
vectorizer, tfidf_matrix = build_tfidf_matrix(
    texts=df["text"],
    max_features=50000
)

tfidf_matrix.shape

(18242, 50000)

## Test queries

We evaluate the TF-IDF baseline on a small fixed set of natural language queries.

The queries were chosen to cover different retrieval situations:
- queries where exact terms should help TF-IDF;
- queries mixing methods and application domains;
- queries where semantic similarity may later help embeddings.

In [12]:
queries_fr = [
    "apprentissage profond pour imagerie médicale",
    "traitement automatique des langues pour documents historiques",
    "apprentissage automatique pour risque de crédit",
    "vision par ordinateur pour diagnostic médical",
    "modèles statistiques pour la finance",
]

queries_en = [
    "deep learning for medical imaging",
    "natural language processing for historical documents",
    "machine learning for credit risk",
]

queries = queries_fr + queries_en

query_language = {query: "fr" for query in queries_fr}
query_language.update({query: "en" for query in queries_en})

queries

['apprentissage profond pour imagerie médicale',
 'traitement automatique des langues pour documents historiques',
 'apprentissage automatique pour risque de crédit',
 'vision par ordinateur pour diagnostic médical',
 'modèles statistiques pour la finance',
 'deep learning for medical imaging',
 'natural language processing for historical documents',
 'machine learning for credit risk']

In [7]:
def display_results(query, top_k=5):
    results = search_tfidf(
        query=query,
        df=df,
        vectorizer=vectorizer,
        matrix=tfidf_matrix,
        top_k=top_k,
    )
    
    cols = ["score", "title", "title_en", "year", "discipline", "subjects", "url"]
    cols = [col for col in cols if col in results.columns]
    
    return results[cols]

## TF-IDF results on French and English queries

The main evaluation queries are written in French because the corpus is mostly composed of French metadata and French abstracts. A smaller set of English queries is also included to test the sensitivity of TF-IDF to query language.

This is important because TF-IDF relies on lexical overlap. An English query can only match a French thesis if English terms are present in the indexed fields, for example in the English title or in bilingual subjects.

In [13]:
all_results = {}

for query in queries:
    all_results[query] = display_results(query, top_k=10)
    print("=" * 100)
    print(query)
    display(all_results[query])

apprentissage profond pour imagerie médicale


,score,title,title_en,year,discipline,subjects,url
5299,0.609534,Apprentissage profond pour l'imagerie médicale 3D,Deep learning for 3D medical imaging,2023.0,Informatique,NaN,https://theses.fr/s389985
6240,0.301117,Perception visuelle computationnelle et cadre ...,Computational Visual Perception and Learning f...,2023.0,Informatique,Quality enhancement ; Low dose CT restoration ...,https://theses.fr/s385753
281,0.239062,Segmentation d'IRM cardiovasculaire utilisant ...,Artificial Intelligence methods applied to car...,2021.0,Instrumentation et informatique de l'image,Segmentation d'images médicales ; Imagerie car...,https://theses.fr/s204634
5686,0.237298,Few-Shot Learning Robuste pour l'Imagerie Médi...,Robust Few-Shot Learning for Medical Imaging,2024.0,Sciences du traitement du signal et des images,Few-shot Learning ; Imaagerie Médicale ; Robus...,https://theses.fr/s412259
10955,0.229963,Apprentissage profond pour la modélisation sta...,NaN,2024.0,Mathematiques,NaN,https://theses.fr/s407925
1122,0.221282,Intelligence Artificielle et Imagerie médicale...,NaN,2023.0,"Recherche clinique, innovation technologique, ...",NaN,https://theses.fr/s392834
5455,0.215092,Une plateforme d'apprentissage profond à base ...,A scalable and component-based deep learning p...,2020.0,Informatique,Génie Logiciel ; Systèmes Distribués ; Hpc ; O...,https://theses.fr/2020GRALM023
1520,0.212911,Représentation uniforme de l'imagerie médicale,Uniform representation of medical imaging,2023.0,Traitement du Signal et des Images,Intelligence artificielle ; Apprentissage prof...,https://theses.fr/2023POIT2258
4979,0.210812,Apprentissage Profond pour l'Inversion d'obser...,NaN,2022.0,"Signal, image et vision",NaN,https://theses.fr/s361492
5825,0.210407,"Quantification de l'incertitude évidencielle, ...",Evidential Quantification of Uncertainty for R...,2025.0,Signal Image Parole Télécoms,Apprentissage Profond Évidentiel ; Quantificat...,https://theses.fr/s427260


traitement automatique des langues pour documents historiques


,score,title,title_en,year,discipline,subjects,url
6742,0.461036,Combiner les séries temporelles et le traiteme...,NaN,2022.0,Sciences de l'information et de la communication,NaN,https://theses.fr/s362025
6686,0.402072,Attention sur les graphes pour le Traitement A...,NaN,2022.0,Informatique,Attention ; Analyse Syntaxique ; Transformers ...,https://theses.fr/s365935
6719,0.356241,La créativité lexicale dans le français contem...,NaN,2015.0,Sciences du langage,NaN,https://theses.fr/s161178
6703,0.339742,Une approche phonétique en identification auto...,NaN,1998.0,Informatique,NaN,https://theses.fr/1998TOU30294
6768,0.326274,Traitement automatique des langues pour la rec...,Natural language processing for music informat...,2020.0,Informatique,Traitement Automatique des Langues ; Segmentat...,https://theses.fr/2020COAZ4017
6700,0.280395,See & Sign : Vision par Ordinateur et Traiteme...,See & Sign: Computer Vision and Natural Langua...,2023.0,Informatique,Langue des signes ; Apprentissage automatique ...,https://theses.fr/s367753
6684,0.279217,Traitement automatique des langues pour l'inde...,Natural language processing for image indexing,2010.0,Informatique,NaN,https://theses.fr/2010REN1S045
6704,0.264889,Variation terminologique et traduction automat...,NaN,2023.0,Sciences du langage - linguistique,NaN,https://theses.fr/s377326
6829,0.256474,Elaboration d'un modèle pour une base de texte...,Elaboration of a model for a pedagogically ind...,2009.0,Sciences du langage,NaN,https://theses.fr/2009GRE39037
6762,0.227720,Biais de genre et traitement automatique des l...,NaN,2024.0,Sciences du langage - linguistique,"Lettres, Sciences Humaines et Sociales",https://theses.fr/s395534


apprentissage automatique pour risque de crédit


...

,score,title,title_en,year,discipline,subjects,url
5299,0.514276,Apprentissage profond pour l'imagerie médicale 3D,Deep learning for 3D medical imaging,2023.0,Informatique,NaN,https://theses.fr/s389985
6365,0.327867,Exploiting Heterogeneous Labels in Deep Learni...,Exploiting Heterogeneous Labels in Deep Learni...,2025.0,Mathématiques et informatique,Apprentissage profond ; Imagerie médicale ; Fo...,https://theses.fr/2025IPPAT017
5371,0.288799,Deep learning for facial micro-expression anal...,Deep learning for facial micro-expression anal...,2022.0,"Signal, Image, Vision",NaN,https://theses.fr/s245776
5196,0.284774,Réseaux antagonistes génératifs appliqués à l'...,Generative adversarial networks for medical im...,2023.0,Analyse et traitement de l'information et des ...,Adaptation de domaine non supervisée ; Contras...,https://theses.fr/2023BRES0010
4880,0.273167,Deep learning for speaker diarization and spee...,NaN,2023.0,Informatique,NaN,https://theses.fr/s376273
4863,0.264463,Ensemble and deep learning for admet properties,NaN,2024.0,Chimie,NaN,https://theses.fr/s393068
4501,0.262660,Deep Learning for Unsupervised Relation Extrac...,Deep Learning for Unsupervised Relation Extrac...,2022.0,Informatique,Apprentissage automatique ; Apprentissage prof...,https://theses.fr/2022SORUS198
5275,0.255092,Deep learning for time series analysis,Deep learning for time series analysis,2023.0,Informatique,NaN,https://theses.fr/s370923
6240,0.252712,Perception visuelle computationnelle et cadre ...,Computational Visual Perception and Learning f...,2023.0,Informatique,Quality enhancement ; Low dose CT restoration ...,https://theses.fr/s385753
4942,0.246810,Deep Learning for Skeletal Character Animation...,NaN,2020.0,Informatique,Département Sciences et technologies de l'info...,https://theses.fr/s361764


natural language processing for historical documents


,score,title,title_en,year,discipline,subjects,url
5072,0.766452,Deep Natural Language Processing for User Repr...,Deep Natural Language Processing for User Repr...,2021.0,Informatique,Deep learning ; Natural language processing ; ...,https://theses.fr/2021SORUS274
3148,0.439156,Neural Transfer Learning for Domain Adaptation...,Neural Transfer Learning for Domain Adaptation...,2021.0,Informatique,Apprentissage par transfert ; Langues et domai...,https://theses.fr/2021UPASG021
6768,0.375605,Traitement automatique des langues pour la rec...,Natural language processing for music informat...,2020.0,Informatique,Traitement Automatique des Langues ; Segmentat...,https://theses.fr/2020COAZ4017
6752,0.264949,A Data-driven Approach to Natural Language Pro...,A Data-driven Approach to Natural Language Pro...,2022.0,Informatique,Modèle de langue ; Corpus de pré-entraînement ...,https://theses.fr/2022SORUS155
6850,0.235702,Surveillance réactive de la mortalité fondée s...,Real-time mortality surveillance based on free...,2019.0,Santé publique - épidémiologie,Traitement du langage ; Causes médicales de dé...,https://theses.fr/2019PESC0100
16502,0.222772,Mieux gérer les communications au sein d'un gr...,Better management of communications in a coope...,2002.0,Informatique,Informatique linguistique ; Communication médi...,https://theses.fr/2002PA112242
6701,0.200148,Natural language processing for subjectivity a...,Natural language processing for subjectivity a...,2026.0,Informatique,Modèles de langue ; Analyse des émotions ; Exp...,https://theses.fr/2026UPASG003
6832,0.192032,Natural language processing of incident and ac...,Natural language processing of incident and ac...,2015.0,Sciences du langage,Traitement automatique des langues ; Retour d'...,https://theses.fr/2015TOU20035
6700,0.190566,See & Sign : Vision par Ordinateur et Traiteme...,See & Sign: Computer Vision and Natural Langua...,2023.0,Informatique,Langue des signes ; Apprentissage automatique ...,https://theses.fr/s367753
16522,0.172902,Analyse automatique des textes pour l'organisa...,Natural language processing for causal organiz...,1998.0,Mathématiques,NaN,https://theses.fr/1998PA040048


machine learning for credit risk


,score,title,title_en,year,discipline,subjects,url
3175,0.321840,Machine learning for performance modelling on ...,Machine learning for performance modelling on ...,2021.0,Informatique,Lignes de Produits Logiciels ; Apprentissage S...,https://theses.fr/2021REN1S117
13184,0.307077,Analyse comportementale du risque de crédit : ...,Behavioural analysis of credit risk : case of ...,2010.0,Sciences de gestion,Risque crédit ; Finance comportementale ; Asym...,https://theses.fr/2010PEST3014
471,0.283801,An EXplainable Artificial Intelligence Credit ...,An EXplainable Artificial Intelligence Credit ...,2023.0,Sciences de l'ingénieur,Risque de crédit ; Machine Learning ; Explicab...,https://theses.fr/2023SORUS486
5064,0.275647,Machine learning for super resolution microscopy,Machine learning for super resolution microscopy,2023.0,Sciences et technologies de l'information et d...,Machine learning ; Dynamique moléculaire ; Adn...,https://theses.fr/s384367
2832,0.275636,Machine learning for the improvement of a blow...,Machine learning for the improvement of a blow...,2022.0,Génie Industriel : Unité de recherche en Mécan...,Caméra thermique ; Soufflage ; Moulage ; Prédi...,https://theses.fr/2022COMP2658
10271,0.248417,Le canal du crédit : une analyse théorique et ...,The credit channel : a theorical and an empiri...,2000.0,Sciences économiques,NaN,https://theses.fr/2000PA122003
5886,0.239015,Risk stratification of hepatocarcinogenesis us...,Risk stratification of hepatocarcinogenesis us...,2024.0,Sciences médicales,Carcinome hépatocellulaire ; Apprentissage pro...,https://theses.fr/2024STRAJ023
18023,0.235090,Construction du processus décisionnel et évalu...,Construction of decision making and assessment...,2011.0,Sciences de gestion,NaN,https://theses.fr/2011PA090077
1685,0.231234,Attribution de crédit pour l'apprentissage par...,Credit Assignment in Deep Reinforcement Learning,2023.0,Mathématiques et informatique,Apprentissage par renforcement ; Attribution d...,https://theses.fr/2023IPPAX155
2551,0.218981,Connecting graphs to machine learning,Connecting graphs to machine learning,2022.0,Informatique,Graphes ; Réseaux ; Citation ; Bipartite ; Par...,https://theses.fr/2022UPSLE018


## Manual relevance assessment

Since the dataset does not provide ground-truth relevance labels, we manually inspect the top retrieved theses.

We use a simple qualitative label:
- `relevant`: the thesis clearly matches the query intent;
- `partial`: the thesis matches only part of the query;
- `irrelevant`: the thesis is not related to the query intent.

In [14]:
manual_eval_rows = []

for query, results in all_results.items():
    top5 = results.head(5).copy()
    for rank, (_, row) in enumerate(top5.iterrows(), start=1):
        manual_eval_rows.append({
            "query": query,
            "language": query_language[query],
            "rank": rank,
            "score": row["score"],
            "title": row["title"],
            "year": row.get("year", None),
            "discipline": row.get("discipline", None),
            "relevance": "",
            "comment": "",
        })

manual_eval = pd.DataFrame(manual_eval_rows)
manual_eval

,query,language,rank,score,title,year,discipline,relevance,comment
0,apprentissage profond pour imagerie médicale,fr,1,0.609534,Apprentissage profond pour l'imagerie médicale 3D,2023.0,Informatique,,
1,apprentissage profond pour imagerie médicale,fr,2,0.301117,Perception visuelle computationnelle et cadre ...,2023.0,Informatique,,
2,apprentissage profond pour imagerie médicale,fr,3,0.239062,Segmentation d'IRM cardiovasculaire utilisant ...,2021.0,Instrumentation et informatique de l'image,,
3,apprentissage profond pour imagerie médicale,fr,4,0.237298,Few-Shot Learning Robuste pour l'Imagerie Médi...,2024.0,Sciences du traitement du signal et des images,,
4,apprentissage profond pour imagerie médicale,fr,5,0.229963,Apprentissage profond pour la modélisation sta...,2024.0,Mathematiques,,
5,traitement automatique des langues pour docume...,fr,1,0.461036,Combiner les séries temporelles et le traiteme...,2022.0,Sciences de l'information et de la communication,,
6,traitement automatique des langues pour docume...,fr,2,0.402072,Attention sur les graphes pour le Traitement A...,2022.0,Informatique,,
7,traitement automatique des langues pour docume...,fr,3,0.356241,La créativité lexicale dans le français contem...,2015.0,Sciences du langage,,
8,traitement automatique des langues pour docume...,fr,4,0.339742,Une approche phonétique en identification auto...,1998.0,Informatique,,
9,traitement automatique des langues pour docume...,fr,5,0.326274,Traitement automatique des langues pour la rec...,2020.0,Informatique,,


In [15]:
manual_labels = {
    # Query 1 - FR
    ("apprentissage profond pour imagerie médicale", 1): (
        "relevant",
        "Direct match: deep learning and 3D medical imaging."
    ),
    ("apprentissage profond pour imagerie médicale", 2): (
        "relevant",
        "Medical imaging and learning framework, especially low-dose CT restoration."
    ),
    ("apprentissage profond pour imagerie médicale", 3): (
        "relevant",
        "Medical image segmentation with AI methods applied to cardiovascular MRI."
    ),
    ("apprentissage profond pour imagerie médicale", 4): (
        "relevant",
        "Few-shot learning directly applied to medical imaging."
    ),
    ("apprentissage profond pour imagerie médicale", 5): (
        "partial",
        "Matches deep learning, but the visible title does not clearly indicate medical imaging."
    ),

    # Query 2 - FR
    ("traitement automatique des langues pour documents historiques", 1): (
        "partial",
        "Matches natural language processing, but the connection to historical documents is unclear."
    ),
    ("traitement automatique des langues pour documents historiques", 2): (
        "partial",
        "Matches NLP and transformers, but not historical documents."
    ),
    ("traitement automatique des langues pour documents historiques", 3): (
        "partial",
        "Related to French language and lexical creativity, but not clearly NLP for historical documents."
    ),
    ("traitement automatique des langues pour documents historiques", 4): (
        "partial",
        "Related to automatic language identification, but not clearly historical documents."
    ),
    ("traitement automatique des langues pour documents historiques", 5): (
        "partial",
        "Strong NLP match, but the application is music information retrieval rather than historical documents."
    ),

    # Query 3 - FR
    ("apprentissage automatique pour risque de crédit", 1): (
        "partial",
        "Strong credit risk match, but the title does not clearly mention machine learning."
    ),
    ("apprentissage automatique pour risque de crédit", 2): (
        "partial",
        "Related to decision process and credit risk assessment, but machine learning is unclear."
    ),
    ("apprentissage automatique pour risque de crédit", 3): (
        "irrelevant",
        "Lexical false positive: credit assignment in reinforcement learning, not credit risk."
    ),
    ("apprentissage automatique pour risque de crédit", 4): (
        "relevant",
        "Direct match: explainable AI, credit scoring, machine learning and credit risk."
    ),
    ("apprentissage automatique pour risque de crédit", 5): (
        "partial",
        "Related to financial risk, but not specifically credit risk or machine learning."
    ),

    # Query 4 - FR
    ("vision par ordinateur pour diagnostic médical", 1): (
        "partial",
        "Matches computer vision, but not medical diagnosis."
    ),
    ("vision par ordinateur pour diagnostic médical", 2): (
        "irrelevant",
        "Human behavior characterization, not clearly medical diagnosis."
    ),
    ("vision par ordinateur pour diagnostic médical", 3): (
        "partial",
        "Matches vision/computer sensing, but not medical diagnosis."
    ),
    ("vision par ordinateur pour diagnostic médical", 4): (
        "partial",
        "Medical and diagnostic terms appear, but the title does not clearly indicate computer vision."
    ),
    ("vision par ordinateur pour diagnostic médical", 5): (
        "partial",
        "Deep learning and explainability are relevant, but medical diagnosis is unclear from the visible fields."
    ),

    # Query 5 - FR
    ("modèles statistiques pour la finance", 1): (
        "partial",
        "Matches statistical models, but the application is NLP semantic frame prediction, not finance."
    ),
    ("modèles statistiques pour la finance", 2): (
        "irrelevant",
        "N-gram language modeling, not finance."
    ),
    ("modèles statistiques pour la finance", 3): (
        "irrelevant",
        "Statistical indicators for genetic sequences and particle physics, not finance."
    ),
    ("modèles statistiques pour la finance", 4): (
        "relevant",
        "Finance-related thesis with probabilistic and statistical modeling."
    ),
    ("modèles statistiques pour la finance", 5): (
        "partial",
        "Matches statistical models, but the domain is pharmacovigilance rather than finance."
    ),

    # Query 6 - EN
    ("deep learning for medical imaging", 1): (
        "relevant",
        "Direct match: deep learning and 3D medical imaging."
    ),
    ("deep learning for medical imaging", 2): (
        "relevant",
        "Direct match: deep learning for medical image analysis and diagnosis."
    ),
    ("deep learning for medical imaging", 3): (
        "partial",
        "Matches deep learning and image analysis, but not medical imaging."
    ),
    ("deep learning for medical imaging", 4): (
        "relevant",
        "Medical imaging and generative adversarial networks."
    ),
    ("deep learning for medical imaging", 5): (
        "partial",
        "Matches deep learning, but the application is speech processing."
    ),

    # Query 7 - EN
    ("natural language processing for historical documents", 1): (
        "partial",
        "Strong NLP match, but not historical documents."
    ),
    ("natural language processing for historical documents", 2): (
        "partial",
        "Strong NLP and transfer learning match, but not historical documents."
    ),
    ("natural language processing for historical documents", 3): (
        "partial",
        "NLP match, but the application is music information retrieval."
    ),
    ("natural language processing for historical documents", 4): (
        "relevant",
        "Directly related to NLP for contemporary and historical French."
    ),
    ("natural language processing for historical documents", 5): (
        "partial",
        "NLP applied to medical causes of death, not historical documents."
    ),

    # Query 8 - EN
    ("machine learning for credit risk", 1): (
        "partial",
        "Matches machine learning, but not credit risk."
    ),
    ("machine learning for credit risk", 2): (
        "partial",
        "Credit risk match, but machine learning is not clear from the visible fields."
    ),
    ("machine learning for credit risk", 3): (
        "relevant",
        "Direct match: AI credit scoring, machine learning and credit risk."
    ),
    ("machine learning for credit risk", 4): (
        "partial",
        "Matches machine learning, but the domain is microscopy."
    ),
    ("machine learning for credit risk", 5): (
        "partial",
        "Matches machine learning, but the application is industrial modeling rather than credit risk."
    ),
}

for idx, row in manual_eval.iterrows():
    key = (row["query"], row["rank"])
    if key in manual_labels:
        manual_eval.loc[idx, "relevance"] = manual_labels[key][0]
        manual_eval.loc[idx, "comment"] = manual_labels[key][1]

manual_eval

,query,language,rank,score,title,year,discipline,relevance,comment
0,apprentissage profond pour imagerie médicale,fr,1,0.609534,Apprentissage profond pour l'imagerie médicale 3D,2023.0,Informatique,relevant,Direct match: deep learning and 3D medical ima...
1,apprentissage profond pour imagerie médicale,fr,2,0.301117,Perception visuelle computationnelle et cadre ...,2023.0,Informatique,relevant,"Medical imaging and learning framework, especi..."
2,apprentissage profond pour imagerie médicale,fr,3,0.239062,Segmentation d'IRM cardiovasculaire utilisant ...,2021.0,Instrumentation et informatique de l'image,relevant,Medical image segmentation with AI methods app...
3,apprentissage profond pour imagerie médicale,fr,4,0.237298,Few-Shot Learning Robuste pour l'Imagerie Médi...,2024.0,Sciences du traitement du signal et des images,relevant,Few-shot learning directly applied to medical ...
4,apprentissage profond pour imagerie médicale,fr,5,0.229963,Apprentissage profond pour la modélisation sta...,2024.0,Mathematiques,partial,"Matches deep learning, but the visible title d..."
5,traitement automatique des langues pour docume...,fr,1,0.461036,Combiner les séries temporelles et le traiteme...,2022.0,Sciences de l'information et de la communication,partial,"Matches natural language processing, but the c..."
6,traitement automatique des langues pour docume...,fr,2,0.402072,Attention sur les graphes pour le Traitement A...,2022.0,Informatique,partial,"Matches NLP and transformers, but not historic..."
7,traitement automatique des langues pour docume...,fr,3,0.356241,La créativité lexicale dans le français contem...,2015.0,Sciences du langage,partial,Related to French language and lexical creativ...
8,traitement automatique des langues pour docume...,fr,4,0.339742,Une approche phonétique en identification auto...,1998.0,Informatique,partial,"Related to automatic language identification, ..."
9,traitement automatique des langues pour docume...,fr,5,0.326274,Traitement automatique des langues pour la rec...,2020.0,Informatique,partial,"Strong NLP match, but the application is music..."


In [16]:
manual_summary = (
    manual_eval
    .groupby(["language", "query", "relevance"])
    .size()
    .unstack(fill_value=0)
    .reset_index()
)

manual_summary

relevance,language,query,irrelevant,partial,relevant
0,en,deep learning for medical imaging,0,2,3
1,en,machine learning for credit risk,0,4,1
2,en,natural language processing for historical doc...,0,4,1
3,fr,apprentissage automatique pour risque de crédit,1,3,1
4,fr,apprentissage profond pour imagerie médicale,0,1,4
5,fr,modèles statistiques pour la finance,2,2,1
6,fr,traitement automatique des langues pour docume...,0,5,0
7,fr,vision par ordinateur pour diagnostic médical,1,4,0


In [17]:
global_summary = manual_eval["relevance"].value_counts().rename_axis("relevance").reset_index(name="count")
global_summary["rate"] = (global_summary["count"] / len(manual_eval) * 100).round(2)

global_summary

,relevance,count,rate
0,partial,25,62.5
1,relevant,11,27.5
2,irrelevant,4,10.0


In [18]:
language_summary = (
    manual_eval
    .groupby(["language", "relevance"])
    .size()
    .unstack(fill_value=0)
)

language_summary["total"] = language_summary.sum(axis=1)

for col in ["relevant", "partial", "irrelevant"]:
    if col in language_summary.columns:
        language_summary[col + "_rate"] = (language_summary[col] / language_summary["total"] * 100).round(2)

language_summary

relevance,irrelevant,partial,relevant,total,relevant_rate,partial_rate,irrelevant_rate
language,,,,,,,
en,0,10,5,15,33.33,66.67,0.0
fr,4,15,6,25,24.00,60.00,16.0
